In [2]:
import os
import json
import torch
import pickle
import numpy as np
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, random_split
from typing import Dict, List, Tuple, Optional, Set
from base import BaseDataLoader, DatasetSubset


In [14]:

import os
import json
import pickle
import torch
import numpy as np
from typing import Tuple, List, Dict, Set, Optional
from torch.utils.data import Dataset, DataLoader

class RedditDataset(Dataset):
    def __init__(
        self, 
        data_dir: str, 
        vocab_path: str,
        split: str = 'train',
        seq_length: int = 10,
        iid: bool = True,
        num_clients: Optional[int] = None,
        client_id: Optional[int] = None,
        clients_root_dir: Optional[str] = None
    ):
        super().__init__()
        self.seq_length = seq_length
        self.split = split
        self.iid = iid
        self.train = split == 'train'
        self.client_id = client_id
        
        # Setup client directory based on number of clients and IID/non-IID
        if clients_root_dir and num_clients:
            distribution_type = "iid" if iid else "non_iid"
            self.clients_data_dir = os.path.join(clients_root_dir, f"{num_clients}_clients_{distribution_type}")
        else:
            self.clients_data_dir = None
        
        # Load vocabulary
        with open(vocab_path, 'rb') as f:
            vocab_dict = pickle.load(f)
            self.vocab = vocab_dict['vocab']
            self.pad_idx = vocab_dict['pad_symbol']
            self.unk_idx = vocab_dict['unk_symbol']
            self.vocab_size = len(self.vocab)

        if self.train:
            if client_id is not None:
                # Load specific client data
                self.sequences = self._load_client_data(client_id)
                self.client_indices = None
            else:
                # Check if data for this configuration exists
                if self.clients_data_dir and os.path.exists(self.clients_data_dir):
                    print(f"Using existing client data from {self.clients_data_dir}")
                    self.sequences = []  # Don't load any sequences in memory
                else:
                    # Initial data processing and client split creation
                    self._process_and_save_client_data(data_dir, num_clients)
                    self.sequences = []
        else:
            # For validation and test sets, check if preprocessed data exists
            split_file = os.path.join(self.clients_data_dir, f'{self.split}_data.pt')
            
            if os.path.exists(split_file):
                print(f"Loading preprocessed {self.split} data from disk")
                self.sequences = torch.load(split_file)
            else:
                print(f"Processing {self.split} data and saving to disk")
                self.sequences, _, _ = self._load_and_prepare_data(data_dir)
                # Create directory if it doesn't exist
                os.makedirs(self.clients_data_dir, exist_ok=True)
                # Save processed data
                torch.save(self.sequences, split_file)

    def _process_and_save_client_data(self, data_dir: str, num_clients: int):
        """Process all data and save client-specific datasets to disk"""
        if not self.clients_data_dir:
            raise ValueError("clients_data_dir must be specified for training data")

        # Create clients directory
        os.makedirs(self.clients_data_dir, exist_ok=True)

        # Load all sequences
        sequences, user_ids, _ = self._load_and_prepare_data(data_dir)

        if self.iid:
            # Create IID split
            indices = list(range(len(sequences)))
            np.random.shuffle(indices)
            
            samples_per_client = len(indices) // num_clients
            remaining_samples = len(indices) % num_clients
            
            current_idx = 0
            
            # Save each client's data separately
            for i in range(num_clients):
                extra_sample = 1 if i < remaining_samples else 0
                client_size = samples_per_client + extra_sample
                
                end_idx = current_idx + client_size
                client_indices = indices[current_idx:end_idx]
                
                # Get client's sequences
                client_sequences = [sequences[idx] for idx in client_indices]
                
                # Save client data to disk
                client_path = os.path.join(self.clients_data_dir, f'client_{i}.pt')
                torch.save(client_sequences, client_path)
                
                current_idx = end_idx
        else:
            # Group sequences by user_id for non-IID
            user_to_sequences = {}
            for seq, user_id in zip(sequences, user_ids):
                if user_id not in user_to_sequences:
                    user_to_sequences[user_id] = []
                user_to_sequences[user_id].append(seq)
            
            # Distribute users among clients
            unique_users = list(user_to_sequences.keys())
            np.random.shuffle(unique_users)
            
            users_per_client = len(unique_users) // num_clients
            remaining_users = len(unique_users) % num_clients
            
            current_idx = 0
            
            # Save each client's data
            for i in range(num_clients):
                extra_user = 1 if i < remaining_users else 0
                client_users_count = users_per_client + extra_user
                
                end_idx = current_idx + client_users_count
                client_users = unique_users[current_idx:end_idx]
                
                # Get all sequences for this client's users
                client_sequences = []
                for user_id in client_users:
                    client_sequences.extend(user_to_sequences[user_id])
                
                # Save client data
                client_path = os.path.join(self.clients_data_dir, f'client_{i}.pt')
                torch.save(client_sequences, client_path)
                
                current_idx = end_idx

    def _load_client_data(self, client_id: int) -> List[torch.Tensor]:
        """Load a specific client's data from disk"""
        if not self.clients_data_dir:
            raise ValueError("clients_data_dir must be specified for loading client data")
            
        client_path = os.path.join(self.clients_data_dir, f'client_{client_id}.pt')
        if not os.path.exists(client_path):
            raise ValueError(f"No data found for client {client_id}")
            
        return torch.load(client_path)

    def _load_and_prepare_data(
        self, 
        data_dir: str,
        num_clients: Optional[int] = None
    ) -> Tuple[List[torch.Tensor], List[str], Optional[Dict[int, Set[int]]]]:
        """Load and prepare data from JSON files"""
        sequences = []
        user_ids = []
        json_files = [f for f in os.listdir(data_dir) 
                     if f.endswith(f'_{self.split}.json')]
        json_files.sort()

        for json_file in json_files:
            with open(os.path.join(data_dir, json_file)) as f:
                data = json.load(f)
                user_data = data['user_data']

                for user_id, user_content in user_data.items():
                    for sequence_list in user_content['x']:
                        if isinstance(sequence_list[0], list):
                            sequence = sequence_list[0]
                        else:
                            sequence = sequence_list

                        token_indices = []
                        for token in sequence:
                            if isinstance(token, str):
                                token_indices.append(self.vocab.get(token, self.unk_idx))
                            else:
                                token_indices.append(self.unk_idx)

                        if len(token_indices) > self.seq_length:
                            token_indices = token_indices[:self.seq_length]
                        elif len(token_indices) < self.seq_length:
                            token_indices.extend([self.pad_idx] * (self.seq_length - len(token_indices)))

                        sequences.append(torch.tensor(token_indices))
                        user_ids.append(user_id)

        return sequences, user_ids, None

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        sequence = self.sequences[idx]
        x = sequence[:-1]
        y = sequence[1:]
        return x, y

    def get_vocab_size(self) -> int:
        return self.vocab_size

    def decode_sequence(self, indices: torch.Tensor) -> List[str]:
        idx_to_token = {idx: token for token, idx in self.vocab.items()}
        return [idx_to_token.get(idx.item(), '<UNK>') for idx in indices]

class RedditDataLoader(BaseDataLoader):
    def __init__(
        self, 
        data_dir: str,
        n_clients: int,
        batch_size: int,
        seq_length: int = 10,
        iid: bool = True
    ):
        super().__init__(data_dir, n_clients, batch_size, iid)
        self.vocab_path = os.path.join(data_dir, 'reddit_vocab.pck')
        self.seq_length = seq_length
        # Setup client data root directory
        self.clients_root_dir = os.path.join(data_dir, 'client_data')

    def load_data(self) -> Tuple[List[DataLoader], DataLoader, DataLoader, int]:
        """Load and prepare the dataset"""
        # Process training data and save client splits if needed
        train_dataset = RedditDataset(
            data_dir=os.path.join(self.data_dir, 'reddit_leaf/train'),
            vocab_path=self.vocab_path,
            split='train',
            seq_length=self.seq_length,
            iid=self.iid,
            num_clients=self.n_clients,
            clients_root_dir=self.clients_root_dir
        )

        val_dataset = RedditDataset(
            data_dir=os.path.join(self.data_dir, 'reddit_leaf/val'),
            vocab_path=self.vocab_path,
            split='val',
            seq_length=self.seq_length,
            clients_root_dir=self.clients_root_dir,
            num_clients=self.n_clients
        )

        test_dataset = RedditDataset(
            data_dir=os.path.join(self.data_dir, 'reddit_leaf/test'),
            vocab_path=self.vocab_path,
            split='test',
            seq_length=self.seq_length,
            clients_root_dir=self.clients_root_dir,
            num_clients=self.n_clients
        )

        # Create empty placeholder for train loaders
        train_loaders = [None] * self.n_clients

        val_loader = DataLoader(
            val_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )
        
        test_loader = DataLoader(
            test_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

        return train_loaders, val_loader, test_loader, train_dataset.get_vocab_size()

    def get_client_dataloader(self, client_id: int) -> DataLoader:
        """Load a specific client's data and create its DataLoader"""
        distribution_type = "iid" if self.iid else "non_iid"
        client_dataset = RedditDataset(
            data_dir=os.path.join(self.data_dir, 'reddit_leaf/train'),
            vocab_path=self.vocab_path,
            split='train',
            seq_length=self.seq_length,
            iid=self.iid,
            client_id=client_id,
            num_clients=self.n_clients,
            clients_root_dir=self.clients_root_dir
        )

        return DataLoader(
            client_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )



In [15]:
import os
import torch
import numpy as np
from collections import Counter
from typing import List, Dict, Tuple



In [16]:
import torch
import numpy as np
from collections import Counter
import os
import json

# Step 1: Initialize the data loader
data_dir = '../data/Reddit'
n_clients = 10
clients_data_dir = os.path.join(data_dir, 'client_data')

data_loader = RedditDataLoader(
    data_dir=data_dir,
    n_clients=n_clients,
    batch_size=32,
    seq_length=10
)

# Get dataloaders (this will process and save client data)
train_loaders, val_loader, test_loader, vocab_size = data_loader.load_data()

Using existing client data from ../data/Reddit/client_data/10_clients_iid
val
../data/Reddit/client_data/10_clients_iid
Processing val data and saving to disk
test
../data/Reddit/client_data/10_clients_iid
Processing test data and saving to disk


In [21]:
# Load client data once
client_loaders = []
client_sizes = []
for i in range(n_clients):
    client_loader = data_loader.get_client_dataloader(i)
    client_loaders.append(client_loader)
    client_sizes.append(len(client_loader.dataset))
    print(f"Client {i}: {len(client_loader.dataset)} samples")

/tmp/ipykernel_154696/3525409817.py:156: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(client_path)


Client 0: 3333118 samples
Client 1: 3333117 samples
Client 2: 3333117 samples
Client 3: 3333117 samples
Client 4: 3333117 samples
Client 5: 3333117 samples
Client 6: 3333117 samples
Client 7: 3333117 samples
Client 8: 3333117 samples
Client 9: 3333117 samples


In [22]:

# 1. Analyze dataset splits
train_size = sum(client_sizes)
val_size = len(val_loader.dataset)
test_size = len(test_loader.dataset)
total_size = train_size + val_size + test_size

print("\n=== Dataset Splits ===")
print(f"{'Train':>12}: {train_size:,} samples ({train_size/total_size:.1%})")
print(f"{'Validation':>12}: {val_size:,} samples ({val_size/total_size:.1%})")
print(f"{'Test':>12}: {test_size:,} samples ({test_size/total_size:.1%})")
print(f"{'Total':>12}: {total_size:,} samples")



=== Dataset Splits ===
       Train: 33,331,171 samples (58.9%)
  Validation: 11,285,200 samples (19.9%)
        Test: 11,970,972 samples (21.2%)
       Total: 56,587,343 samples


In [23]:
# 2. Analyze client distribution
client_stats = {
    'statistics': {
        'min_size': min(client_sizes),
        'max_size': max(client_sizes),
        'mean_size': np.mean(client_sizes),
        'std_size': np.std(client_sizes)
    }
}

print("\n=== Client Distribution ===")
stats = client_stats['statistics']
print(f"Number of Clients: {len(client_sizes)}")
print(f"Samples per Client: {stats['mean_size']:.1f} ± {stats['std_size']:.1f}")
print(f"Range: {stats['min_size']} - {stats['max_size']}")

print("\nClient Distribution Balance:")
for i, size in enumerate(client_sizes):
    print(f"client_{i}: {size/train_size:.1%}")


=== Client Distribution ===
Number of Clients: 10
Samples per Client: 3333117.1 ± 0.3
Range: 3333117 - 3333118

Client Distribution Balance:
client_0: 10.0%
client_1: 10.0%
client_2: 10.0%
client_3: 10.0%
client_4: 10.0%
client_5: 10.0%
client_6: 10.0%
client_7: 10.0%
client_8: 10.0%
client_9: 10.0%


In [24]:
# 3. Analyze sequence lengths using first client
train_lengths = []
pad_idx = client_loaders[0].dataset.pad_idx

# Sample from first client
for batch_x, _ in client_loaders[0]:
    for seq in batch_x:
        length = torch.sum(seq != pad_idx).item()
        train_lengths.append(length)
    break  # Only process first batch

print("\n=== Sequence Length Statistics (sampled from first client) ===")
print(f"Mean Length: {np.mean(train_lengths):.1f} ± {np.std(train_lengths):.1f}")
print(f"Range: {min(train_lengths)} - {max(train_lengths)}")



=== Sequence Length Statistics (sampled from first client) ===
Mean Length: 8.9 ± 0.5
Range: 7 - 9


In [25]:
# 4. Get sequence samples
print("\n=== Sample Sequences ===")
print("\nTraining Samples (from first client):")
for batch_x, _ in client_loaders[0]:
    for i in range(min(3, len(batch_x))):
        sample_x = batch_x[i]
        decoded_x = client_loaders[0].dataset.decode_sequence(sample_x)
        print(f"{i+1}. {' '.join(decoded_x)}")
    break

print("\nValidation Samples:")
for batch_x, _ in val_loader:
    for i in range(min(3, len(batch_x))):
        sample_x = batch_x[i]
        decoded_x = val_loader.dataset.decode_sequence(sample_x)
        print(f"{i+1}. {' '.join(decoded_x)}")
    break


=== Sample Sequences ===

Training Samples (from first client):
1. <BOS> in person is going to be a whole
2. <BOS> god i love this i would enjoy seeing
3. <BOS> gotta give the young man some credit ...

Validation Samples:
1. <BOS> im all onboard with you guys in principle
2. <BOS> and they didnt finish filming until not too
3. <BOS> we crash to 35 in the fv before


In [28]:

# 5. If using non-IID, analyze user distribution from metadata
if not data_loader.iid:
    print("\n=== User Distribution (Non-IID) ===")
    train_dir = os.path.join(data_dir, 'reddit_leaf/train')
    json_files = [f for f in os.listdir(train_dir) if f.endswith('_train.json')]
    
    sequences_per_user = Counter()
    for json_file in json_files:
        with open(os.path.join(train_dir, json_file)) as f:
            data = json.load(f)
            for user_id, user_content in data['user_data'].items():
                sequences_per_user[user_id] += len(user_content['x'])
    
    user_seq_counts = list(sequences_per_user.values())
    print(f"Total Users: {len(sequences_per_user)}")
    print("\nSequences per User:")
    print(f"Mean: {np.mean(user_seq_counts):.1f} ± {np.std(user_seq_counts):.1f}")
    print(f"Range: {min(user_seq_counts)} - {max(user_seq_counts)}")

# Clean up at the very end
for loader in client_loaders:
    del loader

In [29]:

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer

class RedditTransformer(nn.Module):
    def __init__(self, vocab_size, emsize=400, nhead=8, nhid=400, 
                 nlayers=4, dropout=0.2, max_seq_length=64):
        super(RedditTransformer, self).__init__()
        self.model_type = 'Transformer'
        self.emsize = emsize
        self.src_mask = None
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, emsize)
        self.pos_encoder = PositionalEncoding(emsize, dropout, max_seq_length)
        
        # Transformer encoder
        encoder_layers = TransformerEncoderLayer(
            d_model=emsize,
            nhead=nhead,
            dim_feedforward=nhid,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = TransformerEncoder(encoder_layers, nlayers)
        
        # Decoder to predict next token
        self.decoder = nn.Linear(emsize, vocab_size)
        
        self.init_weights()

    def generate_square_subsequent_mask(self, sz):
        """Generate mask for autoregressive prediction"""
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def init_weights(self):
        initrange = 0.1
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.decoder.bias.data.zero_()
        self.decoder.weight.data.uniform_(-initrange, initrange)

    def forward(self, src, src_padding_mask=None):
        """
        Args:
            src: Tensor, shape [batch_size, seq_len]
            src_padding_mask: Optional tensor indicating which elements in src are padding
                            shape [batch_size, seq_len]
        Returns:
            output: Tensor containing log probabilities for next token prediction
                   shape [batch_size, seq_len, vocab_size]
        """
        if self.src_mask is None or self.src_mask.size(0) != src.size(1):
            mask = self.generate_square_subsequent_mask(src.size(1))
            self.src_mask = mask.to(src.device)
            
        # Embedding and positional encoding
        src = self.embedding(src) * math.sqrt(self.emsize)  # [batch_size, seq_len, emsize]
        src = self.pos_encoder(src)
        
        # Transformer encoder with masking for autoregressive prediction
        output = self.transformer_encoder(
            src, 
            mask=self.src_mask,
            src_key_padding_mask=src_padding_mask
        )
        
        # Decode to get next token predictions
        output = self.decoder(output)  # [batch_size, seq_len, vocab_size]
        
        return output

    def get_attention_weights(self):
        """Returns attention weights from all layers if available"""
        weights = []
        for layer in self.transformer_encoder.layers:
            if hasattr(layer.self_attn, 'attention_weights'):
                weights.append(layer.self_attn.attention_weights)
        return weights

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        
        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        Args:
            x: Tensor, shape [batch_size, seq_len, embedding_dim]
        """
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy
import math
from tqdm import tqdm

def train_federated(
    model,
    data_loader,
    criterion,
    optimizer,
    device,
    n_rounds=5,
    local_epochs=1,
    clip_grad=0.25,
    log_interval=100
):
    """
    Train model using federated learning with memory-efficient data loading
    
    Args:
        model: RedditTransformer model
        data_loader: RedditDataLoader instance
        criterion: Loss function
        optimizer: Optimizer
        device: torch device
        n_rounds: Number of communication rounds
        local_epochs: Number of local epochs per client
        clip_grad: Gradient clipping value
        log_interval: Logging interval
    """
    # Get initial dataloaders
    _, val_loader, test_loader, vocab_size = data_loader.load_data()
    
    # Initialize tracking variables
    best_val_loss = float('inf')
    best_model = None
    train_losses = []
    val_losses = []
    
    # Global rounds
    for round_idx in range(n_rounds):
        print(f'\nCommunication Round {round_idx+1}/{n_rounds}')
        
        round_client_losses = []
        client_states = []
        
        # Keep copy of global model parameters
        global_model_state = copy.deepcopy(model.state_dict())
        
        # Train each client sequentially
        for client_idx in range(data_loader.n_clients):
            print(f'\nTraining Client {client_idx+1}/{data_loader.n_clients}')
            
            # Get client's dataloader
            client_loader = data_loader.get_client_dataloader(client_idx)
            
            # Reset client model to global model state
            model.load_state_dict(global_model_state)
            
            # Train client for multiple local epochs
            client_losses = []
            for local_epoch in range(local_epochs):
                print(f'Local Epoch {local_epoch+1}/{local_epochs}')
                train_loss, train_ppl = train_epoch(
                    model=model,
                    train_loader=client_loader,
                    optimizer=optimizer,
                    criterion=criterion,
                    device=device,
                    clip_grad=clip_grad,
                    log_interval=log_interval
                )
                client_losses.append(train_loss)
                
            avg_client_loss = np.mean(client_losses)
            round_client_losses.append(avg_client_loss)
            print(f'Client {client_idx+1} Average Loss: {avg_client_loss:.4f}')
            
            # Save client's model parameters
            client_states.append(copy.deepcopy(model.state_dict()))
            
            # Clear client's dataloader to free memory
            del client_loader
            torch.cuda.empty_cache()
        
        # Aggregate client models (FedAvg)
        with torch.no_grad():
            # Average model parameters
            for key in model.state_dict().keys():
                averaged_param = torch.stack([state[key].float() 
                                           for state in client_states], 0).mean(0)
                model.state_dict()[key].copy_(averaged_param)
        
        # Calculate average loss for this round
        round_loss = np.mean(round_client_losses)
        train_losses.append(round_loss)
        
        # Evaluate global model on validation set
        val_loss, val_ppl = evaluate(
            model=model,
            data_loader=val_loader,
            criterion=criterion,
            device=device
        )
        val_losses.append(val_loss)
        
        print(f'\nRound {round_idx+1} Summary:')
        print(f'Average Train Loss: {round_loss:.4f}')
        print(f'Validation Loss: {val_loss:.4f}, Validation PPL: {val_ppl:.2f}')
        
        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model = copy.deepcopy(model)
            print('New best model saved!')
        
        # Clear memory
        del client_states
        torch.cuda.empty_cache()
    
    return best_model, train_losses, val_losses

def train_epoch(model, train_loader, optimizer, criterion, device, clip_grad, log_interval):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    total_tokens = 0
    
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Move to device
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        # Create padding mask
        padding_mask = (inputs == criterion.ignore_index)
        
        # Forward pass
        optimizer.zero_grad()
        output = model(inputs, src_padding_mask=padding_mask)
        
        # Reshape output and targets for loss computation
        output = output.view(-1, output.size(-1))
        targets = targets.view(-1)
        
        # Compute loss
        loss = criterion(output, targets)
        
        # Backward pass
        loss.backward()
        
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        
        # Update weights
        optimizer.step()
        
        # Calculate statistics
        non_pad_mask = targets != criterion.ignore_index
        num_tokens = non_pad_mask.sum().item()
        
        total_loss += loss.item() * num_tokens
        total_tokens += num_tokens
        
        # Print progress
        if batch_idx % log_interval == 0:
            print(f'Batch {batch_idx}/{len(train_loader)}, '
                  f'Loss: {loss.item():.4f}, '
                  f'PPL: {math.exp(loss.item()):.2f}')
    
    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    
    return avg_loss, perplexity

def evaluate(model, data_loader, criterion, device):
    """Evaluate model on given dataloader"""
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            # Create padding mask
            padding_mask = (inputs == criterion.ignore_index)
            
            # Forward pass
            output = model(inputs, src_padding_mask=padding_mask)
            
            # Reshape for loss computation
            output = output.view(-1, output.size(-1))
            targets = targets.view(-1)
            
            # Compute loss
            loss = criterion(output, targets)
            
            # Calculate statistics
            non_pad_mask = targets != criterion.ignore_index
            num_tokens = non_pad_mask.sum().item()
            
            total_loss += loss.item() * num_tokens
            total_tokens += num_tokens
    
    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    
    return avg_loss, perplexity

# Example usage:



In [31]:

# Initialize data loader
data_loader = RedditDataLoader(
    data_dir='../data/Reddit',
    n_clients=10,
    batch_size=32,
    seq_length=10,
    iid=True
)

# Get vocabulary size
_, _, _, vocab_size = data_loader.load_data()
print(f'Vocabulary Size: {vocab_size}')





Using existing client data from ../data/Reddit/client_data/10_clients_iid
val
../data/Reddit/client_data/10_clients_iid
Loading preprocessed val data from disk


/tmp/ipykernel_154696/3525409817.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.sequences = torch.load(split_file)


test
../data/Reddit/client_data/10_clients_iid
Loading preprocessed test data from disk
Vocabulary Size: 49999


TypeError: string indices must be integers, not 'str'

In [34]:
# Initialize model and move to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RedditTransformer(vocab_size=vocab_size, max_seq_length=10).to(device)
# Initialize criterion and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=val_loader.dataset.pad_idx)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train model
best_model, train_losses, val_losses = train_federated(
    model=model,
    data_loader=data_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device
)

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
